# NHL Game Data — Gap Fill: Star Schema, Missing Silver Tables & Test Suite
**Run this notebook AFTER RL_Notebook_Fixed.**

| Section | What it adds |
|---------|-------------|
| 1 | Missing Silver tables (skater stats, goalie stats, game plays) |
| 2 | Star Schema Gold layer (dim_team, dim_player, dim_date, fact tables) |
| 3 | Gold KPI views (standings, player seasons, home/away, period goals, power play) |
| 4 | Formal test suite (row counts, nulls, FK integrity, business logic, range checks) |

---
## Setup & Imports

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.shuffle.partitions', '8')
spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')

print('Spark ready:', spark.version)
print('Existing tables:')
spark.sql('SHOW TABLES').show(50, truncate=False)

---
## Section 1 — Missing Silver Tables

In [ ]:
# 1.1 silver_game_skater_stats
silver_game_skater_stats = (
    spark.table('game_skater_stats')
    .dropDuplicates(['game_id', 'player_id'])
    .withColumn('shooting_pct',
        F.round(F.when(F.col('shots') > 0, F.col('goals') / F.col('shots') * 100)
                 .otherwise(0), 2))
    .withColumn('points', F.col('goals') + F.col('assists'))
    .withColumn('toi_minutes', F.round(F.col('timeOnIce') / 60, 2))
    .na.fill(0, subset=['goals', 'assists', 'shots', 'hits', 'blocked',
                        'penaltyMinutes', 'giveaways', 'takeaways', 'plusMinus'])
)
silver_game_skater_stats.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('silver_game_skater_stats')
print(f'silver_game_skater_stats: {silver_game_skater_stats.count():,} rows saved')

In [ ]:
# 1.2 silver_game_goalie_stats
silver_game_goalie_stats = (
    spark.table('game_goalie_stats')
    .dropDuplicates(['game_id', 'player_id'])
    .withColumn('save_pct',
        F.round(F.when(F.col('shots') > 0, F.col('saves') / F.col('shots'))
                 .otherwise(None), 3))
    .withColumn('goals_against_avg',
        F.round(F.when(F.col('timeOnIce') > 0,
                       F.col('goals') / (F.col('timeOnIce') / 3600))
                 .otherwise(None), 2))
    .na.fill(0, subset=['goals', 'saves', 'shots', 'pim',
                        'evenSaves', 'powerPlaySaves', 'shortHandedSaves'])
)
silver_game_goalie_stats.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('silver_game_goalie_stats')
print(f'silver_game_goalie_stats: {silver_game_goalie_stats.count():,} rows saved')

In [ ]:
# 1.3 silver_game_plays
silver_game_plays = (
    spark.table('game_plays')
    .dropDuplicates(['play_id'])
    .withColumn('period_type',
        F.when(F.col('period') <= 3, 'regulation')
         .when(F.col('period') == 4, 'overtime')
         .otherwise('shootout'))
    .na.fill({'x': 0.0, 'y': 0.0, 'secondaryType': 'N/A', 'description': ''})
)
silver_game_plays.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('silver_game_plays')
print(f'silver_game_plays: {silver_game_plays.count():,} rows saved')

---
## Section 2 — Star Schema Gold Layer

In [ ]:
# Load silver tables
game        = spark.table('silver_game')
teams_stats = spark.table('silver_team_game_stats_valid')
skater      = spark.table('silver_game_skater_stats')
goalie      = spark.table('silver_game_goalie_stats')
plays       = spark.table('silver_game_plays')
player_info = spark.table('silver_player')
team_info   = spark.table('silver_team')
print('All silver tables loaded.')

In [ ]:
# 2.1 dim_team
dim_team = (
    team_info
    .select(
        F.col('team_id'),
        F.col('franchiseId').alias('franchise_id'),
        F.col('shortName').alias('city'),
        F.col('teamName').alias('team_name'),
        F.col('abbreviation'),
        F.col('link').alias('nhl_api_link')
    )
    .dropDuplicates(['team_id'])
)
dim_team.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('dim_team')
print(f'dim_team: {dim_team.count()} rows saved')

In [ ]:
# 2.2 dim_player
dim_player = (
    player_info
    .select(
        F.col('player_id'),
        F.col('firstName').alias('first_name'),
        F.col('lastName').alias('last_name'),
        F.concat(F.col('firstName'), F.lit(' '), F.col('lastName')).alias('full_name'),
        F.col('nationality'),
        F.col('birthCity').alias('birth_city'),
        F.col('primaryPosition').alias('position'),
        F.col('birthDate').alias('birth_date'),
        F.col('weight').alias('weight_lbs'),
        F.col('shootsCatches').alias('shoots_catches')
    )
    .dropDuplicates(['player_id'])
)
dim_player.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('dim_player')
print(f'dim_player: {dim_player.count():,} rows saved')

In [ ]:
# 2.3 dim_date
dim_date = (
    game.select(F.col('date_time_gmt')).dropna()
    .withColumn('date_key', F.date_format('date_time_gmt', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('date',         F.to_date('date_time_gmt'))
    .withColumn('year',         F.year('date_time_gmt'))
    .withColumn('month',        F.month('date_time_gmt'))
    .withColumn('month_name',   F.date_format('date_time_gmt', 'MMMM'))
    .withColumn('day',          F.dayofmonth('date_time_gmt'))
    .withColumn('day_of_week',  F.dayofweek('date_time_gmt'))
    .withColumn('day_name',     F.date_format('date_time_gmt', 'EEEE'))
    .withColumn('quarter',      F.quarter('date_time_gmt'))
    .withColumn('week_of_year', F.weekofyear('date_time_gmt'))
    .withColumn('is_weekend',
        F.when(F.dayofweek('date_time_gmt').isin([1, 7]), True).otherwise(False))
    .select('date_key', 'date', 'year', 'month', 'month_name', 'day',
            'day_of_week', 'day_name', 'quarter', 'week_of_year', 'is_weekend')
    .dropDuplicates(['date_key'])
    .orderBy('date_key')
)
dim_date.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('dim_date')
print(f'dim_date: {dim_date.count():,} rows saved')

In [ ]:
# 2.4 fact_game_performance
# Reload silver tables — safe to run even if cell-5 was skipped
game        = spark.table('silver_game')
teams_stats = spark.table('silver_team_game_stats_valid')

# FIX: only faceOffWinPercentage exists (pre-calculated) — no raw faceOffWins/faceoffTaken
# FIX: fill 8 source nulls on goals/shots with 0
teams_stats_enriched = (
    teams_stats
    .na.fill(0, ['goals', 'shots', 'hits', 'pim',
                 'powerPlayOpportunities', 'powerPlayGoals',
                 'giveaways', 'takeaways', 'blocked'])
    .withColumn('shot_pct',
        F.round(
            F.when(F.col('shots') > 0, F.col('goals') / F.col('shots') * 100)
             .otherwise(0), 2))
    .withColumn('faceoff_win_pct',
        F.round(F.col('faceOffWinPercentage'), 2))
)

fact_game_performance = (
    teams_stats_enriched
    .join(
        game.select('game_id', 'season', 'type', 'date_time_gmt',
                    'home_team_id', 'away_team_id', 'outcome'),
        'game_id', 'left'
    )
    .withColumn('date_key',
        F.date_format('date_time_gmt', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('opponent_team_id',
        F.when(F.col('HoA') == 'home', F.col('away_team_id'))
         .otherwise(F.col('home_team_id')))
    .select(
        'game_id', 'team_id', 'opponent_team_id',
        'season', 'type', 'date_key',
        F.col('HoA').alias('home_or_away'),
        'won', 'settled_in',
        'goals', 'shots', 'hits', 'pim',
        F.col('powerPlayOpportunities'),
        F.col('powerPlayGoals'),
        'giveaways', 'takeaways', 'blocked',
        'shot_pct', 'faceoff_win_pct',
        'head_coach'
    )
)

fact_game_performance.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('fact_game_performance')
print(f'fact_game_performance: {fact_game_performance.count():,} rows saved')

In [ ]:
# 2.5 fact_player_stats — skaters + goalies unioned
# Reload — safe to run standalone
game   = spark.table('silver_game')
skater = spark.table('silver_game_skater_stats')
goalie = spark.table('silver_game_goalie_stats')

game_keys = game.select('game_id', 'season', 'type', 'date_time_gmt')

skater_fact = (
    skater
    .join(game_keys, 'game_id', 'left')
    .withColumn('date_key', F.date_format('date_time_gmt', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('player_type', F.lit('skater'))
    .select(
        'game_id', 'player_id', 'team_id', 'season', 'type', 'date_key', 'player_type',
        F.col('goals').cast(IntegerType()),
        F.col('assists').cast(IntegerType()),
        F.col('points').cast(IntegerType()),
        F.col('shots').cast(IntegerType()),
        F.col('hits').cast(IntegerType()),
        F.col('blocked').cast(IntegerType()),
        F.col('penaltyMinutes').alias('penalty_minutes').cast(IntegerType()),
        F.col('plusMinus').alias('plus_minus').cast(IntegerType()),
        'giveaways', 'takeaways', 'toi_minutes', 'shooting_pct',
        F.lit(None).cast(DoubleType()).alias('save_pct'),
        F.lit(None).cast(DoubleType()).alias('goals_against_avg')
    )
)

goalie_fact = (
    goalie
    .join(game_keys, 'game_id', 'left')
    .withColumn('date_key', F.date_format('date_time_gmt', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('player_type', F.lit('goalie'))
    .select(
        'game_id', 'player_id', 'team_id', 'season', 'type', 'date_key', 'player_type',
        F.lit(0).cast(IntegerType()).alias('goals'),
        F.lit(0).cast(IntegerType()).alias('assists'),
        F.lit(0).cast(IntegerType()).alias('points'),
        F.col('shots').cast(IntegerType()),
        F.lit(0).cast(IntegerType()).alias('hits'),
        F.lit(0).cast(IntegerType()).alias('blocked'),
        F.col('pim').alias('penalty_minutes').cast(IntegerType()),
        F.lit(None).cast(IntegerType()).alias('plus_minus'),
        F.lit(0).alias('giveaways'),
        F.lit(0).alias('takeaways'),
        F.round(F.col('timeOnIce') / 60, 2).alias('toi_minutes'),
        F.lit(None).cast(DoubleType()).alias('shooting_pct'),
        'save_pct', 'goals_against_avg'
    )
)

fact_player_stats = skater_fact.union(goalie_fact)
fact_player_stats.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('fact_player_stats')
print(f'fact_player_stats: {fact_player_stats.count():,} rows saved')

In [ ]:
# 2.6 fact_play_events
# Reload — safe to run standalone
game  = spark.table('silver_game')
plays = spark.table('silver_game_plays')

KEY_EVENTS = ['Goal', 'Shot', 'Penalty', 'Hit', 'Missed Shot',
              'Blocked Shot', 'Takeaway', 'Giveaway', 'Faceoff']

fact_play_events = (
    plays
    .filter(F.col('event').isin(KEY_EVENTS))
    .join(game.select('game_id', 'season', 'type', 'date_time_gmt'), 'game_id', 'left')
    .withColumn('date_key', F.date_format('date_time_gmt', 'yyyyMMdd').cast(IntegerType()))
    .select(
        'play_id', 'game_id', 'season', 'type', 'date_key',
        F.col('team_id_for'), F.col('team_id_against'),
        'event', F.col('secondaryType').alias('secondary_type'),
        'period', 'period_type', F.col('periodTime').alias('period_time'),
        'x', 'y', 'description'
    )
)
fact_play_events.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('fact_play_events')
print(f'fact_play_events: {fact_play_events.count():,} rows saved')

---
## Section 3 — Gold KPI Views

In [ ]:
# 3.1 gold_team_standings
spark.sql("""
    SELECT
        f.season,
        t.abbreviation, t.team_name, t.city,
        COUNT(DISTINCT f.game_id)                                            AS games_played,
        SUM(CAST(f.won AS INT))                                              AS wins,
        COUNT(DISTINCT f.game_id) - SUM(CAST(f.won AS INT))                 AS losses,
        ROUND(SUM(CAST(f.won AS INT)) / COUNT(DISTINCT f.game_id) * 100, 1) AS win_pct,
        SUM(f.goals)                                                         AS goals_for,
        SUM(f.shots)                                                         AS total_shots,
        ROUND(AVG(f.shot_pct), 2)                                            AS avg_shot_pct,
        ROUND(AVG(f.faceoff_win_pct), 2)                                     AS avg_faceoff_win_pct,
        SUM(f.powerPlayGoals)                                                AS pp_goals,
        SUM(f.powerPlayOpportunities)                                        AS pp_opportunities,
        ROUND(SUM(f.powerPlayGoals) / NULLIF(SUM(f.powerPlayOpportunities), 0) * 100, 2)
                                                                             AS pp_efficiency_pct,
        SUM(f.hits)                                                          AS total_hits,
        SUM(f.giveaways)                                                     AS total_giveaways,
        SUM(f.takeaways)                                                     AS total_takeaways
    FROM fact_game_performance f
    JOIN dim_team t ON f.team_id = t.team_id
    WHERE f.type = 'R'
    GROUP BY f.season, t.abbreviation, t.team_name, t.city
    ORDER BY f.season, win_pct DESC
""").write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_team_standings')
print('gold_team_standings saved')

In [ ]:
# 3.2 gold_player_season_stats
spark.sql("""
    SELECT
        ps.season, ps.player_id, p.full_name, p.position, p.nationality,
        t.abbreviation AS team, ps.player_type,
        COUNT(DISTINCT ps.game_id)          AS games_played,
        SUM(ps.goals)                       AS goals,
        SUM(ps.assists)                     AS assists,
        SUM(ps.goals) + SUM(ps.assists)     AS points,
        SUM(ps.shots)                       AS shots,
        SUM(ps.hits)                        AS hits,
        SUM(ps.blocked)                     AS blocked_shots,
        SUM(ps.penalty_minutes)             AS penalty_minutes,
        SUM(ps.plus_minus)                  AS plus_minus,
        ROUND(AVG(ps.toi_minutes), 2)       AS avg_toi_minutes,
        ROUND(AVG(ps.shooting_pct), 2)      AS avg_shooting_pct,
        ROUND(AVG(ps.save_pct), 3)          AS avg_save_pct,
        ROUND(AVG(ps.goals_against_avg), 2) AS avg_gaa
    FROM fact_player_stats ps
    JOIN dim_player p ON ps.player_id = p.player_id
    JOIN dim_team   t ON ps.team_id   = t.team_id
    WHERE ps.type = 'R'
    GROUP BY ps.season, ps.player_id, p.full_name, p.position,
             p.nationality, t.abbreviation, ps.player_type
    ORDER BY ps.season, points DESC
""").write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_player_season_stats')
print('gold_player_season_stats saved')

In [ ]:
# 3.3 gold_home_away_summary
spark.sql("""
    SELECT
        season, home_or_away,
        COUNT(game_id)                                           AS total_games,
        SUM(CAST(won AS INT))                                   AS wins,
        ROUND(SUM(CAST(won AS INT)) / COUNT(game_id) * 100, 1)  AS win_pct,
        ROUND(AVG(goals), 2)            AS avg_goals,
        ROUND(AVG(shots), 2)            AS avg_shots,
        ROUND(AVG(hits), 2)             AS avg_hits,
        ROUND(AVG(faceoff_win_pct), 2)  AS avg_faceoff_win_pct,
        ROUND(AVG(shot_pct), 2)         AS avg_shot_pct
    FROM fact_game_performance
    WHERE type = 'R'
    GROUP BY season, home_or_away
    ORDER BY season, home_or_away
""").write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_home_away_summary')
print('gold_home_away_summary saved')

In [ ]:
# 3.4 gold_goals_by_period
spark.sql("""
    SELECT season, period, period_type,
           secondary_type AS shot_type,
           COUNT(*)       AS total_goals
    FROM fact_play_events
    WHERE event = 'Goal'
    GROUP BY season, period, period_type, secondary_type
    ORDER BY season, period
""").write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_goals_by_period')
print('gold_goals_by_period saved')

In [ ]:
# 3.5 gold_powerplay_efficiency
spark.sql("""
    SELECT f.season, t.abbreviation, t.team_name,
           SUM(f.powerPlayOpportunities) AS pp_opportunities,
           SUM(f.powerPlayGoals)         AS pp_goals,
           ROUND(SUM(f.powerPlayGoals) / NULLIF(SUM(f.powerPlayOpportunities), 0) * 100, 2)
                                         AS pp_efficiency_pct
    FROM fact_game_performance f
    JOIN dim_team t ON f.team_id = t.team_id
    WHERE f.type = 'R'
    GROUP BY f.season, t.abbreviation, t.team_name
    ORDER BY f.season, pp_efficiency_pct DESC
""").write.format('delta').mode('overwrite') \
    .option('overwriteSchema', True).saveAsTable('gold_powerplay_efficiency')
print('gold_powerplay_efficiency saved')

In [ ]:
print('=== All Tables ===')
spark.sql('SHOW TABLES').show(100, truncate=False)

---
## Section 4 — Formal Test Suite

In [ ]:
test_results = []

def test(name: str, passed: bool, detail: str = ''):
    status = 'PASS' if passed else 'FAIL'
    icon   = '✅' if passed else '❌'
    test_results.append({'test': name, 'status': status, 'detail': detail})
    print(f'{icon} {status}  |  {name}  {"— " + detail if detail else ""}')

def check_no_nulls(df, table_name, columns):
    for c in columns:
        n = df.filter(F.col(c).isNull()).count()
        test(f'{table_name}.{c} has no nulls', n == 0,
             f'{n} nulls found' if n > 0 else '')

print('Test framework ready.')

In [ ]:
silver_game_t  = spark.table('silver_game')
fact_game_t    = spark.table('fact_game_performance')
fact_player_t  = spark.table('fact_player_stats')
fact_plays_t   = spark.table('fact_play_events')
dim_team_t     = spark.table('dim_team')
dim_player_t   = spark.table('dim_player')
dim_date_t     = spark.table('dim_date')

sg_cnt  = silver_game_t.count()
fgp_cnt = fact_game_t.count()
print(f'silver_game={sg_cnt:,}  fact_game_performance={fgp_cnt:,}')

In [ ]:
print('--- Row Count Checks ---')
test('fact_game_performance ~2x silver_game',
     abs(fgp_cnt - sg_cnt * 2) < sg_cnt * 0.05,
     f'silver_game={sg_cnt:,}, fact_game={fgp_cnt:,}')
test('dim_team >= 30 teams',     dim_team_t.count() >= 30,    f'{dim_team_t.count()} rows')
test('dim_player >= 1000',       dim_player_t.count() >= 1000, f'{dim_player_t.count():,} rows')
test('dim_date >= 365 dates',    dim_date_t.count() >= 365,   f'{dim_date_t.count():,} rows')
test('fact_play_events >= 100k', fact_plays_t.count() >= 100000, f'{fact_plays_t.count():,} rows')
test('fact_player_stats >= 10k', fact_player_t.count() >= 10000, f'{fact_player_t.count():,} rows')

In [ ]:
print('--- Null Checks ---')
check_no_nulls(dim_team_t,   'dim_team',   ['team_id', 'abbreviation', 'team_name'])
check_no_nulls(dim_player_t, 'dim_player', ['player_id', 'full_name', 'position'])
check_no_nulls(dim_date_t,   'dim_date',   ['date_key', 'date', 'year', 'month'])
check_no_nulls(fact_game_t,  'fact_game_performance', ['game_id', 'team_id', 'season', 'goals', 'shots'])
check_no_nulls(fact_player_t,'fact_player_stats',     ['game_id', 'player_id', 'team_id', 'player_type'])

In [ ]:
print('--- Referential Integrity ---')
test('fact_game.team_id in dim_team',
     fact_game_t.join(dim_team_t, 'team_id', 'left_anti').count() == 0)
test('fact_player.player_id in dim_player',
     fact_player_t.join(dim_player_t, 'player_id', 'left_anti').count() == 0)
test('fact_game.date_key in dim_date',
     fact_game_t.join(dim_date_t, 'date_key', 'left_anti').count() == 0)

# Some historical/relocated franchises (e.g. Atlanta Thrashers) are in skater stats
# but not in the team reference table — treat <=5 orphan teams as acceptable
_orphan_teams = (fact_player_t
                 .join(dim_team_t, 'team_id', 'left_anti')
                 .select('team_id').distinct().count())
test('fact_player.team_id in dim_team (<=5 orphan teams acceptable)',
     _orphan_teams <= 5,
     f'{_orphan_teams} orphan team_id(s) — historical/relocated franchises')

In [ ]:
print('--- Business Logic ---')
test('goals non-negative', fact_game_t.filter(F.col('goals') < 0).count() == 0)
test('shot_pct 0-100',
     fact_game_t.filter((F.col('shot_pct') < 0) | (F.col('shot_pct') > 100)).count() == 0)
test('save_pct 0-1',
     fact_player_t.filter(F.col('save_pct').isNotNull())
                  .filter((F.col('save_pct') < 0) | (F.col('save_pct') > 1)).count() == 0)
test('each game has 2 team rows',
     fact_game_t.groupBy('game_id').agg(F.count('team_id').alias('n'))
                .filter(F.col('n') != 2).count() == 0)
test('home_or_away only home/away',
     fact_game_t.filter(~F.col('home_or_away').isin(['home','away'])).count() == 0)

In [ ]:
print('--- Range Checks ---')
max_g = fact_game_t.agg(F.max('goals')).collect()[0][0]
test('max goals per game <= 15', max_g <= 15, f'max={max_g}')

# Dataset covers seasons from 2000-01 through 2019-20, encoded as e.g. 20002001
seasons = [r['season'] for r in fact_game_t.select('season').distinct().collect()]
test('seasons in valid range',
     all(19992000 <= s <= 20232024 for s in seasons),
     str(sorted(seasons)))

# Periods: 1-3 regulation, 4 OT, 5 shootout; playoffs can have OT periods 6, 7, 8+
_bad_periods = fact_plays_t.filter(F.col('period') < 1).count()
test('period values >= 1 (playoffs may exceed period 5)',
     _bad_periods == 0,
     f'{_bad_periods} rows with period < 1')

test('shot coords within rink bounds',
     fact_plays_t.filter(F.col('event').isin(['Shot','Goal']))
                 .filter((F.col('x') < -110) | (F.col('x') > 110) |
                         (F.col('y') < -50)  | (F.col('y') > 50)).count() == 0)

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)

df_r   = pd.DataFrame(test_results)
passed = (df_r['status'] == 'PASS').sum()
failed = (df_r['status'] == 'FAIL').sum()

print(f'\n{"="*60}')
print(f'  TEST SUMMARY: {passed}/{len(df_r)} passed  |  {failed} failed')
print(f'{"="*60}\n')
print(df_r[['test','status','detail']].to_string(index=False))

if failed > 0:
    print(f'\n⚠️  {failed} test(s) FAILED — review before presenting.')
else:
    print('\n🎉 All tests passed — data is ready for Power BI!')